In [0]:

from pyspark.sql import functions as F

catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema = "hospital_project"

patients = spark.table(
    f"`{catalog}`.{schema}.silver_patients"
)

appointments = spark.table(
    f"`{catalog}`.{schema}.silver_appointments"
)

treatments = spark.table(
    f"`{catalog}`.{schema}.silver_treatments"
)

print("Patients:", patients.count())
print("Appointments:", appointments.count())
print("Treatments:", treatments.count())


Patients: 50
Appointments: 90
Treatments: 43


In [0]:

patient_summary = patients.agg(
    F.countDistinct("patient_id").alias("total_patients")
)

appointment_summary = appointments.agg(
    F.count("*").alias("total_appointments"),
    F.sum(
        F.when(F.col("status") == "Completed", 1)
         .otherwise(0)
    ).alias("completed_appointments"),
    F.sum(
        F.when(F.col("status") == "Cancelled", 1)
         .otherwise(0)
    ).alias("cancelled_appointments")
)

treatment_summary = treatments.agg(
    F.count("*").alias("total_treatments"),
    F.sum("cost").alias("total_treatment_revenue")
)

gold_hospital_summary = (
    patient_summary
    .crossJoin(appointment_summary)
    .crossJoin(treatment_summary)
)

display(gold_hospital_summary)


total_patients,total_appointments,completed_appointments,cancelled_appointments,total_treatments,total_treatment_revenue
50,90,43,21,43,72850.00


In [0]:

gold_hospital_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"`{catalog}`.{schema}.gold_hospital_summary"
    )

print("Hospital summary Gold table created!")


Hospital summary Gold table created!


In [0]:

# Aggregate treatment costs for each appointment
treatment_totals = (
    treatments
    .groupBy("appointment_id")
    .agg(
        F.count("*").alias("treatment_count"),
        F.sum("cost").alias("treatment_revenue")
    )
)

# Join appointment records with treatment totals
department_source = (
    appointments
    .join(
        treatment_totals,
        on="appointment_id",
        how="left"
    )
)

# Department-level business metrics
gold_department_analytics = (
    department_source
    .groupBy("department")
    .agg(
        F.count("*").alias("total_appointments"),
        F.sum(
            F.when(
                F.col("status") == "Completed", 1
            ).otherwise(0)
        ).alias("completed_appointments"),
        F.countDistinct("patient_id").alias("unique_patients"),
        F.sum(
            F.coalesce(F.col("treatment_count"), F.lit(0))
        ).alias("total_treatments"),
        F.sum(
            F.coalesce(
                F.col("treatment_revenue"),
                F.lit(0)
            )
        ).alias("total_treatment_revenue")
    )
)

display(gold_department_analytics)


department,total_appointments,completed_appointments,unique_patients,total_treatments,total_treatment_revenue
Orthopedics,19,8,15,8,16700.00
General Medicine,21,9,18,9,10950.00
Cardiology,16,7,13,7,10700.00
Dermatology,19,10,16,10,17500.00
Neurology,15,9,14,9,17000.00


In [0]:

gold_department_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"`{catalog}`.{schema}.gold_department_analytics"
    )

print("Department analytics Gold table created!")


Department analytics Gold table created!


In [0]:

gold_daily_appointments = (
    appointments
    .groupBy("appointment_date")
    .agg(
        F.count("*").alias("total_appointments"),
        F.sum(
            F.when(
                F.col("status") == "Completed", 1
            ).otherwise(0)
        ).alias("completed_appointments")
    )
    .orderBy("appointment_date")
)

display(gold_daily_appointments)


appointment_date,total_appointments,completed_appointments
2026-06-01,3,2
2026-06-04,1,1
2026-06-05,1,1
2026-06-06,1,0
2026-06-07,1,1
2026-06-08,2,0
2026-06-09,2,1
2026-06-10,3,1
2026-06-12,1,0
2026-06-14,2,1


In [0]:

gold_daily_appointments.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"`{catalog}`.{schema}.gold_daily_appointments"
    )

print("Daily appointments Gold table created!")


Daily appointments Gold table created!


In [0]:

gold_tables = [
    "gold_hospital_summary",
    "gold_department_analytics",
    "gold_daily_appointments"
]

for table in gold_tables:
    df = spark.table(
        f"`{catalog}`.{schema}.{table}"
    )
    print(f"{table}: {df.count()} rows")

# Important reconciliation checks
department_df = spark.table(
    f"`{catalog}`.{schema}.gold_department_analytics"
)

assert department_df.agg(
    F.sum("total_appointments")
).first()[0] == appointments.count()

assert department_df.agg(
    F.sum("total_treatments")
).first()[0] == treatments.count()

print("Gold reconciliation checks passed!")


gold_hospital_summary: 1 rows
gold_department_analytics: 5 rows
gold_daily_appointments: 56 rows
Gold reconciliation checks passed!
